In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os
from scipy.stats import gaussian_kde
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, Ellipse
import matplotlib.colors as mcolors
from matplotlib import colors
import cmaps


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

path = r'/Volumes/shiki/study/data/SST/202609/'
path_noaa = r'../../../study/data/noaa-20c/'

In [ ]:
sst_2026 = xr.DataArray(np.empty((12, 180, 360)), coords=[np.arange(1, 13), np.arange(89.5, -90, -1), np.arange(-179.5, 180, 1)], dims=['month', 'lat', 'lon'])
f = open(r'/Users/jiangdi/Downloads/HadISST1_SST_2026.txt')
data_str = f.readlines()
n = -1

for i in range(len(data_str)):
    data = data_str[i][:-1].replace('-', ' -').split()
    if data[-1] == 'columns':
        n += 1
        rows = -1
        # print(data)
    else:
        if len(data) == 360:
            rows += 1
            sst_2026[n, rows, :] = np.array(data, dtype=np.int64)
sst_2026 = xr.concat([sst_2026.loc[:, :, 0:], sst_2026.loc[:, :, :0]], dim='lon').assign_coords(lon=np.arange(0.5, 360, 1))
sst_2026 = sst_2026.where(np.logical_and(abs(sst_2026) > 1000, abs(sst_2026) < 10000))/100
sst_2026 = sst_2026.interp(lat=np.arange(-40, 41, 1), lon=np.arange(0, 360, 1))

sst = xr.open_dataset('./data/sst.nc').sst.loc['1980':'2025', -40:40, 0:359]
ssta_2026 = sst_2026[:7] - sst.groupby('time.month').mean()[:7]

tsurfa = xr.open_dataset(path+'tsurfa.9.adj.1sep2026.24m.nc').tsurfa.mean(dim='lev')
tsurfa = tsurfa.interp(lat=np.arange(-40, 41, 1), lon=np.arange(0, 360, 1))
mask = np.isnan(ssta_2026[0])
tsurfa = tsurfa.where(~mask)

time = pd.date_range('2026-01', '2028-08', freq='MS')
lat = np.arange(-40, 41, 1)
lon = np.arange(0, 360, 1)
nt, ny, nx = len(time), len(lat), len(lon)
ssta_2027 = xr.DataArray(np.empty((nt, ny, nx)), coords={'time':time, 'lat':lat, 'lon':lon})
ssta_2027[:7, :, :] = ssta_2026.values
ssta_2027[8:, :, :] = tsurfa.values
ssta_2027[7] = (ssta_2027[6] + ssta_2027[8])/2


In [ ]:
sst = xr.open_dataset('./data/sst.nc').sst.loc['1980':'2025', -40:40, 0:359]
ssta_2026 = sst_2026[:7] - sst.groupby('time.month').mean()[:7]

In [ ]:
tsurfa = xr.open_dataset(path+'tsurfa.9.adj.1sep2026.24m.nc').tsurfa.mean(dim='lev')
tsurfa = tsurfa.interp(lat=np.arange(-40, 41, 1), lon=np.arange(0, 360, 1))
mask = np.isnan(ssta_2026[0])
tsurfa = tsurfa.where(~mask)

In [ ]:
time = pd.date_range('2026-01', '2028-08', freq='MS')
lat = np.arange(-40, 41, 1)
lon = np.arange(0, 360, 1)
nt, ny, nx = len(time), len(lat), len(lon)
ssta_2027 = xr.DataArray(np.empty((nt, ny, nx)), coords={'time':time, 'lat':lat, 'lon':lon})
ssta_2027[:7, :, :] = ssta_2026.values
ssta_2027[8:, :, :] = tsurfa.values
ssta_2027[7] = (ssta_2027[6] + ssta_2027[8])/2

In [ ]:
ssta_2027.loc[:, -5:5, :].mean('lat').plot.contourf(levels=np.linspace(-2, 2, 21), cmap=cmaps.hotcold_18lev)

In [ ]:
precp = xr.open_dataset('/Users/jiangdi/Downloads/precip.mon.mean.nc', decode_times=False).precip
precp = precp.assign_coords(time=precp.time.values * np.timedelta64(1, 'D') + np.datetime64('1800-01-01', 'D'))
prea = precp.groupby('time.month') - precp.groupby('time.month').mean()
prea = prea.interp(lat=lat, lon=lon)
prea_pre = xr.open_dataset(path+'tprepa.9.adj.1sep2026.24m.nc').tprepa.mean(dim='lev').interp(lat=lat, lon=lon)
prea_2027 = xr.DataArray(np.empty((nt, ny, nx)), coords={'time':time, 'lat':lat, 'lon':lon})

prea_2027[:6] = prea[-6:].values
prea_2027[8:] = prea_pre.values
prea_2027[6:8] = np.nan
prea_2027 = prea_2027.interpolate_na(dim="time", method="linear")

In [ ]:
da = prea_2027
month = da.time.dt.month

lat_nh = (da.lat >= 5) & (da.lat <= 25) 
lat_sh = (da.lat >= -25) & (da.lat <= -5)
is_jun_nov = (month >= 6) & (month <= 11)
is_dec_may = (month >= 12) | (month <= 5)

mask = (is_jun_nov & lat_nh) | (is_dec_may & lat_sh)

prea_hov = prea_2027.where(mask).mean('lat')

In [ ]:

prea_hov.rolling(time=3, lon=4).mean().plot.contourf(levels=np.linspace(-2, 2, 21), cmap=cmaps.hotcold_18lev)

In [ ]:
ds_wnd = xr.open_dataset('/Users/jiangdi/Downloads/wnd.nc').rename({'valid_time':'time', 'pressure_level':'level', 'latitude':'lat', 'longitude':'lon'})
uwnd = ds_wnd.u.interp(lat=lat, lon=lon)
vwnd = ds_wnd.v.interp(lat=lat, lon=lon)
wwnd = ds_wnd.w.loc[:, 500]

In [ ]:
d = 6.371e+6 * np.pi / 180
dudz_2026 = uwnd.loc[:, 200] - uwnd.loc[:, 850]
vort_2026 = (vwnd.sel(level=850).differentiate('lon') - uwnd.sel(level=850).differentiate('lat'))/d

In [ ]:
uwnd = xr.open_dataset(path_noaa+'uwnd.mon.mean.tp.nc').uwnd.loc['1970':'2010', [850, 200]]
vwnd = xr.open_dataset(path_noaa+'vwnd.mon.mean.tp.nc').vwnd.loc['1970':'2010', [850, 200]]
vort = (vwnd.loc[:, 850].differentiate('lon')  - uwnd.loc[:, 850].differentiate('lat'))/ d
dudz = uwnd.loc[:, 200] - uwnd.loc[:, 850]

In [ ]:
vort_2026 = vort_2026.groupby('time.month') - vort.groupby('time.month').mean()
dudz_2026 = dudz_2026.groupby('time.month') - dudz.groupby('time.month').mean()


In [ ]:
u_pre = xr.open_dataset(path+'U850a.9.adj.1sep2026.24m.nc').u850a.mean(dim='lev').interp(lat=lat, lon=lon)
v_pre = xr.open_dataset(path+'V850a.9.adj.1sep2026.24m.nc').v850a.mean(dim='lev').interp(lat=lat, lon=lon)

vort_pre = (v_pre.differentiate('lon')  - u_pre.differentiate('lat'))/ d
vort_2027 = xr.DataArray(np.empty((nt, ny, nx)), coords={'time':time, 'lat':lat, 'lon':lon})

vort_2027[:9] = vort_2026.values
vort_2027[9:] = vort_pre[1:].values

vort_2027.loc[:, :0, :] = -vort_2027.loc[:, :0, :]
vort_hov = vort_2027.where(mask).mean('lat')


In [ ]:
vort_hov.rolling(time=3, lon=5).mean().plot.contourf(levels=np.linspace(-1, 1, 21)*1e-5, cmap=cmaps.hotcold_18lev)

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(4, 8))
ssta_hov = ssta_2027.loc[:, -5:5].mean(dim='lat')
axs[0].contourf(lon, time[:-5], ssta_hov[:-5], levels=np.linspace(-2, 2, 21), cmap=cmaps.hotcold_18lev, extend='both')
axs[0].set_title('SSTA')
axs[1].contourf(lon, time[:-5], prea_hov[:-5].rolling(time=3, lon=5, center=True, min_periods=1).mean(), levels=np.linspace(-2, 2, 21), cmap=cmaps.CBR_drywet, extend='both')
axs[1].set_title('Preciptation')
cb = axs[2].contourf(lon, time[:-5], vort_hov[:-5].rolling(time=3, lon=5, center=True, min_periods=1).mean()*4e+5, levels=np.linspace(-2, 2, 21), cmap=cmaps.hotcold_18lev, extend='both')
axs[2].set_title('Vort')
# for i in range(1, 3):
#     ax = axs[i]
#     ax.add_patch(Ellipse(xy=(0.6, 0.4), width=0.7, height=0.2, angle=55, linewidth=1.5,
#                      edgecolor='red', facecolor='none', alpha=0.8, transform=ax.transAxes))
cbaxes = fig.add_axes([0.2, 0.04, 0.6, 0.01])
cb = plt.colorbar(cb, cax=cbaxes, orientation="horizontal")
fig.subplots_adjust(left=0.07, right=0.93, bottom=0.1, top=0.9, wspace=0.4, hspace=0.5)